## 1. Dataset 정의
- CSV에서 path, slope_avg를 읽어 PyTorch가 바로 쓸 수 있는 샘플 단위 객체로 만드는 단계

### Dataset?
- CSV 한 줄을 읽어서, 모델이 바로 먹을 수 있는 샘플 1개로 바꿔주는 객체
- PyTorch 공식 튜토리얼도 custom dataset은 보통 __init__, __len__, __getitem__ 세 가지를 구현한다고 설명
  * [PyTorchDocs](https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html?utm_source=chatgpt.com)
- DataSet은 단순하게 유지, **샘플 하나를 깔끔하게 꺼내는 역할**
#### 목적
- 데이터 로딩을 표준화하기 위해서
- DataLoader와의 연결

#### python에서의 class
- `__init__` : 객체가 만들어질 때 처음 실행되는 초기화 함수
- `__len__` : 데이터가 총 몇개인지 반환합니다.
- `__getitem__(idx)` : `idx`번째 샘플 하나를 반환합니다. 
- 앞뒤의 __는 Python이 내부적으로 사용하는 규약.
- len, genitem은 우리가 만든 일반 함수가 아니라 Python의 "특수 메서드(special method)"를 오버라이딩한 것

In [ ]:
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset


# df : DataFrame
# transform : 생성한 transform
class SlopeDataset(Dataset): # PyTorch Dataset을 상속
    def __init__(self, data, transform=None): # 초기화 함수
        if isinstance(data, pd.DataFrame):
            self.df = data.reset_index(drop=True).copy()
        else:
            self.df = pd.read_csv(data)
            
        self.transform = transform
    
    def __len__(self): # 전체 샘플 수를 반환
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = Path(row["path"])
        image = Image.open(image_path).convert("RGB")

        target = torch.tensor(row["slope_avg"], dtype=torch.float32)

        if self.transform is not None:
            image = self.transform(image)

        return image, target

## 2. Transform 설계
- 이미지를 ConvNeXt 입력 규격에 맞게 변환하는 단계

### ConvNeXt-Tiny pretrained weights
- ConvNeXt-Tiny는 pretrained weight는 공식적으로 `resize=236 → center crop=224 → [0,1] 스케일 → ImageNext mean/std normalize` 전처리를 사용
- 또한 torchvision은 pretrained 모델마다 맞는 전처리를 weights 객체와 함께 제공하며 이를 따르지 않으면 성능이 떨어질 수 있다고 명시함
  * [PyTorch convnext_tiny Docs](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.convnext_tiny.html)

### transform?
- 원본 이미지를 모델이 학습하기 좋은 형태로 바꾸는 규칙 모음
- 이미지 크기 맞추기, 텐서로 변경, 정규화 등


### WheelSafe Project는?
- ImageNet 분류가 아니라 도로 경사 회귀이므로, 학습용 transform은 ConvNeXt의 입력 규격을 맞추되, 경사 단서(원근, 바닥 패턴, 화면의 수직 구조)를 망가뜨리지 않는 약한 augmentation로 설계하는 것이 맞음
- 일반적인 augmentation 연구도 augmentation의 목적을 "훈련 분포와 실제 분포의 차이를 줄이는 것"으로 설명하며, 교통 도로 도메인 리뷰는 특히 조명 변화, 가림, 실제 도로 환경 변화 대응에는 도움이 되지만, 도메인 의미를 꺠는 변형은 부적절하다고 시사
  * [sciencedirect](https://www.sciencedirect.com/science/article/pii/S2590005622000911)
- Albumentations는 강력하지만 2차 고도화에 고려
- 강한 augmentation으로 부족한 데이터를 억지로 늘리는 전략”보다, “약한 augmentation으로 실제 촬영 변동만 흉내내는 전략”이 더 적절
- 교통 시각 도메인에서는 조명 변화, 일부 가림, 날씨/광량 차이 같은 현실 변동을 견디는 것이 중요하다.
  * 교통 시각 요소에 대한 최근 리뷰도 augmentation이 실제 환경 변화 대응과 robustness에 중요하다고 정리
  * 따라서 색/밝기 관련 augmentation은 유효할 가능성이 

#### Resize는?
- 공식 규격은 **resize 236 → center crop 224 → normalize** 이다.
- 하지만 WProject는 **화면 전체의 도로 구조**까 중요하다.
- train은 Resize(236, 236) 뒤 RandomCrop(224, 224)로 적용, 이유: 약한 위치 변화만 주고, 과한 왜곡은 막기 위함
- valid/test는 Resize(236, 236) 뒤 CenterCrop(224, 224), 이유: 평가를 안정적으로 고정

#### Horizontal Flip은?
- 낮은 확률로 허용
- 전반 경사(오르막/내리막) 자체는 좌우 반전으로 바뀌지 않음
- 좌우 시각 편향을 약간 줄일 수 있음
- 확률은 0.3 정도로 보수적으로 잡음

#### Rotation은?
- 작은 rotation은 카메라의 미세한 기울어짐을 흉내낼 수 있지만, 큰 rotation은 경사 단서를 망가뜨릴 수 있음
- 허용 : +- 3정도

#### Brightness / Contrast는 왜 필요?
- 도로/보행 도메인에서는 햇빛 강도, 그림자, 노출 차이, 시간대 차이 등의 변화가 흔하다.
- ColorJitter(brightness=0.1, contrast=0.1)
- 색조, 채도는 굳이 넣지 않는다.

#### Nomalize는 왜 꼭 해야 하나?
- 입력 분포를 맞춰주는 단계
- ConvNeXt -pretrained 모델은 Image Net 기준 mean/std 정규화를 기대함

#### torchvision으로 갈까, albumentations로 갈까?
- torchvision으로 감
- 공식 문서상 classification 포함 다양한 비전 작업에 강력한 augmentation 파이프라인을 제공하지만, 지금은 단순성과 연결성이 더 중요

**성능보다 먼저 라벨 의미를 안전하게 보존하는 transform이 우선**

In [ ]:
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

RESIZE_SIZE = 236
CROP_SIZE = 224


def get_train_transform():
    return transforms.Compose([
        transforms.Resize((RESIZE_SIZE, RESIZE_SIZE)),
        transforms.RandomCrop((CROP_SIZE, CROP_SIZE)),
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.RandomApply(
            [transforms.ColorJitter(brightness=0.1, contrast=0.1)],
            p=0.5,
        ),
        transforms.RandomRotation(degrees=3),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def get_eval_transform():
    return transforms.Compose([
        transforms.Resize((RESIZE_SIZE, RESIZE_SIZE)),
        transforms.CenterCrop((CROP_SIZE, CROP_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

## 3. Split 정의
- 전체 데이터를 train / valid / test로 나누는 규칙

### Group Slit이 필요한 이유?
- WProject의 데이터는 단일 독립 이미지들의 모음이 아니라 같은 장소 / 같은 촬영 구간 / 연속 프레임이 섞여 있을 가능성이 크다.
- 이런 상황은 data leakage 또는 overly optimistic evaluation로 이어질 수 있다
  * scikit-learn도 test 정보가 학습 과정에 섞이면 과도하게 낙관적인 점수가 나온다고 경고하고 있음
  * [scikit-learn](https://scikit-learn.org/stable/common_pitfalls.html?utm_source=chatgpt.com)
- GroupShuffleSplit group 단위로 train/test를 나누기 때문에, 같은 group이 양쪽에 동시에 들어가지 않도록 도와줌

### 우리는 어떤 group을 사용?
- folder, 왜? 같은 폴더 안 이미지가 같은 촬영 세션/구간일 가능성이 높기 때문
### 왜 GroupShuffleSplit을 쓰는가?
- (차후 보강) GroupKFold와의 차이, 비교.

In [ ]:
from pathlib import Path
from lib.utils.path import output_path

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

csv_path = output_path() / 'slope_labels_20260309_185927.csv'
group_col="folder"

train_size=0.7
valid_size=0.15
test_size=0.15

df = pd.read_csv(csv_path)

if group_col not in df.columns:
    raise ValueError(f"'{group_col}' column not found in CSV.")

if abs(train_size + valid_size + test_size - 1.0) > 1e-8:
    raise ValueError("train_size, valid_size, test_size must sum to 1.0.")

groups = df[group_col]

train_splitter = GroupShuffleSplit(
    n_splits=1,
    train_size=train_size,
    random_state=42,
)
train_idx, temp_idx = next(train_splitter.split(df, groups=groups))

train_df = df.iloc[train_idx].reset_index(drop=True)
temp_df = df.iloc[temp_idx].reset_index(drop=True)

temp_groups = temp_df[group_col]
valid_ratio_in_temp = valid_size / (valid_size + test_size) # 처음에 train을 먼저 떼어내고 남은 temp안에서 valid와 test를 다시 나눠야 한다.

valid_splitter = GroupShuffleSplit(
    n_splits=1,
    train_size=valid_ratio_in_temp,
    random_state=42,
)
valid_idx, test_idx = next(valid_splitter.split(temp_df, groups=temp_groups))

valid_df = temp_df.iloc[valid_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

## 4. DataLoader 정의
- Dataset에서 샘플을 여러 개 꺼내서 batch로 묶어 주는 도구
### batch를 쓰는 이유?
- GPU의 효율적으로 사용
- gradient를 더 안정적으로 계산
- 학습 속도 상승

### pin_memory?
- GPU 학습할 때 CPU → GPU 메모리 전송을 조금 더 효율적으로 도와주는 옵션

### drop_last?
- 마지막 batch가 너무 작으면 버리는 옵션

In [ ]:
from torch.utils.data import DataLoader

train_dataset = SlopeDataset(train_df, transform=get_train_transform())
valid_dataset = SlopeDataset(valid_df, transform=get_eval_transform())
test_dataset = SlopeDataset(test_df, transform=get_eval_transform())

batch_size = 32
num_workers = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)

### 이해하면 좋은 shape 개념
#### image 1장
- transform 적용 후 보통:
``` Python
[3, 224, 224]
```

- batch로 묶이면 적용 후 보통:
``` Python
[batch_size,3, 224, 224]
```

## 5. Model 정의
- 입력 이미지에서 패턴을 읽고, 최종 예측값을 만드는 함수

### 이미지 회귀(image regression)
- 이미지 feature를 추출하고
- 그 feature를 바탕으로
- 실수값 1개를 예측하는 것

### backbone과 head
- backbone
  * 이미지에서 feature를 추출하는 본체
  * edge, texture, road boundary, perspective cue, slope-related scene structure 이런 걸 점점 추상적인 feature로 바꾼다.(무슨 뜻입니까?)
- head
  * backbone이 만든 feature를 최종 출력으로 바꾸는 마지막 부분
### 설계 목표
- ConvNeXt-Tiny pretrained 사용
- 마지막 classifier만 회귀용으로 교체
- 출력 shape은 [B]
- 현재 transform / DataLoader 흐름과 바로 연결
- 나중에 loss, optimizer, train loop와 연결하기 쉬운 구조

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights


class ConvNeXtTinyRegressor(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()

        weights = ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
        self.model = convnext_tiny(weights=weights)

        in_features = self.model.classifier[2].in_features # 마지막 classifier layer가 받는 feature 차원을 읽어온다.
        self.model.classifier[2] = nn.Linear(in_features, 1) # 기존 분류용 layer을 회귀용 출력 1개 layer로 변경

    def forward(self, x):
        x = self.model(x)
        return x.squeeze(1) # 모델 원래 출력은 보통 [B, 1] => [B]로 정리

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ConvNeXtTinyRegressor(pretrained=True).to(device)
print(model)

In [ ]:
import torch.nn as nn

criterion = nn.HuberLoss(delta=1.0)

## 7. Optimizer & Scheduler
- loss를 줄이기 위해 모델의 파라미터를 실제로 업데이트하는 도구
### 왜 Optimizer?
- 딥러닝 모델은 단순히 loss만 정의한다고 학습되지 않는다. loss는 "얼마나 틀렸는지"를 알려줄 뿐이고,
실제로 파라미터를 바꾸는 건 optimizer이다.
### 왜 AdamW?
- weight decay가 momentum/variance와 분리된(decoupled) 형태의 Adam 계열 optimizer로 제공
- pretrained vision backbone fine-tuning에 매우 무난함
- 학습이 잘 시작됨
- weight decay를 함께 쓰기 좋음
- ConvNeXt 같은 현대적 비전 모델 fine-tuning에서 자주 쓰임
### weight decay?
- 모델이 너무 과하게 특정 파라미터에 의존하지 않도록 약한 규제를 주는 것
- 과적합 완화에 도움
- pretrained fine-tuning에서 안정감 부여

## Scheduler
- 학습률(learning rate)을 학습 과정에서 조절하는 도구
### 왜, CosineAnnealingLR?
- 학습률을 cosine 곡선 형태로 점진적으로 줄이는 scheduler
* 구현이 단순함, epoch 기반 학습과 잘 맞음, 초반에는 비교적 크게, 후반에는 작게 줄여감, 첫 버전에서 무난하고 안정적
### T_max가 뭐예요?
- cosine 주기가 끝나는 시점을 의미, scheduler의 최대 iteration 수
- epoch 단위로 scheduler를 쓸 가능성이 높으므로, `T_max = num_epochs` (?)
- `eta_min`은 scheduler가 줄여 나갈 최소 learning rate, 학습률이 0까지 떨어지지 않게 바닥값을 주는 역할 (?)
### 설계
- optimizer는 AdamW
- scheduler는 CosineAnnealingLR

In [ ]:
import torch

num_epochs = 20

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=1e-6,
)

## Train, Validation 정의
### Train Loop가 뭔가요?
- **모델이 실제로 배우는 반복문**
1. 이미지를 모델에 넣는다
2. 예측값을 만든다
3. loss를 계산한다
4. backward로 gradient를 계산한다
5. optimizer로 파라미터를 업데이트한다

### Validation은 뭔가요?
- **모델이 지금 얼마나 일반화되고 있는지 확인하는 평가 단계**
  * 학습은 하지 않음
  * 파라미터 업데이트 없음, gradient 계산 없음, 성능만 측정

### 왜 validation?
- 학습만 할 경우
  * train loss만 보고 있음
  * 과적합을 모름
  * 어느 epoch가 가장 좋은지 모름
  * checkpoint 기준을 세우기 어려움

In [ ]:
from tqdm.auto import tqdm
import torch


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_abs_error = 0.0
    total_sq_error = 0.0
    total_count = 0

    for images, targets in tqdm(loader, desc="Train", leave=False):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()

        preds = model(images)
        loss = criterion(preds, targets)

        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_abs_error += torch.abs(preds - targets).sum().item()
        total_sq_error += torch.square(preds - targets).sum().item()
        total_count += batch_size

    epoch_loss = total_loss / total_count
    epoch_mae = total_abs_error / total_count
    epoch_rmse = (total_sq_error / total_count) ** 0.5

    return epoch_loss, epoch_mae, epoch_rmse


def validate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_abs_error = 0.0
    total_sq_error = 0.0
    total_count = 0

    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Valid", leave=False):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            preds = model(images)
            loss = criterion(preds, targets)

            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            total_abs_error += torch.abs(preds - targets).sum().item()
            total_sq_error += torch.square(preds - targets).sum().item()
            total_count += batch_size

    epoch_loss = total_loss / total_count
    epoch_mae = total_abs_error / total_count
    epoch_rmse = (total_sq_error / total_count) ** 0.5

    return epoch_loss, epoch_mae, epoch_rmse

In [ ]:
best_val_mae = float("inf")

for epoch in range(num_epochs):
    train_loss, train_mae, train_rmse = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
    )

    val_loss, val_mae, val_rmse = validate(
        model=model,
        loader=valid_loader,
        criterion=criterion,
        device=device,
    )

    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"lr: {current_lr:.6f} | "
        f"train_loss: {train_loss:.4f} | train_mae: {train_mae:.4f} | train_rmse: {train_rmse:.4f} | "
        f"val_loss: {val_loss:.4f} | val_mae: {val_mae:.4f} | val_rmse: {val_rmse:.4f}"
    )

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        print(f"Best model updated. val_mae = {best_val_mae:.4f}")